In [2]:
!pip install rdkit
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

import lightgbm as lgb
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
print('All imports successful.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 53.0 MB/s eta 0:00:00:00:0100:01
All imports successful.


In [3]:
import glob
import pandas as pd

train_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
test_path  = glob.glob('/kaggle/input/**/test.csv', recursive=True)[0]

print("Train:", train_path)
print("Test :", test_path)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
print(f'Train: {train.shape}  |  Test: {test.shape}')
print('\nTarget type counts (train):')
print(train['target_type'].value_counts())
print('\nTarget type counts (test):')
print(test['target_type'].value_counts())

train_tg  = train[train['target_type'] == 'tg'].reset_index(drop=True)
train_egc = train[train['target_type'] == 'egc'].reset_index(drop=True)
test_tg   = test[test['target_type'] == 'tg'].reset_index(drop=True)
test_egc  = test[test['target_type'] == 'egc'].reset_index(drop=True)

y_tg  = train_tg['target'].values
y_egc = train_egc['target'].values

print(f'\nTg  train: {len(train_tg):,}  test: {len(test_tg):,}')
print(f'Egc train: {len(train_egc):,}  test: {len(test_egc):,}')
print(f'Tg  range: [{y_tg.min():.1f}, {y_tg.max():.1f}]')
print(f'Egc range: [{y_egc.min():.4f}, {y_egc.max():.4f}]')

Train: /kaggle/input/datasets/josephjonathanfdes/anrf-aisehack2-0/train.csv
Test : /kaggle/input/datasets/josephjonathanfdes/anrf-aisehack2-0/test.csv
Train: (6171, 3)  |  Test: (4115, 3)

Target type counts (train):
target_type
tg     4143
egc    2028
Name: count, dtype: int64

Target type counts (test):
target_type
tg     2763
egc    1352
Name: count, dtype: int64

Tg  train: 4,143  test: 2,763
Egc train: 2,028  test: 1,352
Tg  range: [-118.0, 490.0]
Egc range: [0.1032, 9.8627]


In [4]:
DESC_NAMES = [n for n, _ in Descriptors.descList]
MORGAN_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def featurize(smiles_list):
    rdkit_rows, morgan_rows, maccs_rows = [], [], []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rdkit_rows.append([np.nan] * len(DESC_NAMES))
            morgan_rows.append(np.zeros(2048, dtype=np.uint8))
            maccs_rows.append(np.zeros(167, dtype=np.uint8))
        else:
            vals = Descriptors.CalcMolDescriptors(mol)
            rdkit_rows.append(list(vals.values()))
            morgan_rows.append(MORGAN_GEN.GetFingerprintAsNumPy(mol))
            fp_mac = MACCSkeys.GenMACCSKeys(mol)
            maccs_rows.append(np.array(fp_mac, dtype=np.uint8))
    rdkit_df  = pd.DataFrame(rdkit_rows,  columns=DESC_NAMES)
    morgan_df = pd.DataFrame(morgan_rows, columns=[f'morgan_{i}' for i in range(2048)])
    maccs_df  = pd.DataFrame(maccs_rows,  columns=[f'maccs_{i}'  for i in range(167)])
    return pd.concat([rdkit_df, morgan_df, maccs_df], axis=1)


def build_preprocessor(X_raw):
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    thresh = int(0.2 * len(X))
    X = X.dropna(axis=1, thresh=thresh)
    X = X.loc[:, X.var() > 0]
    good_cols = X.columns.tolist()
    imputer = SimpleImputer(strategy='median')
    X_imp   = imputer.fit_transform(X)
    scaler  = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)
    return X_scaled, (imputer, scaler, good_cols)


def apply_preprocessor(X_raw, preprocessor):
    imputer, scaler, good_cols = preprocessor
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.reindex(columns=good_cols, fill_value=np.nan)
    return scaler.transform(imputer.transform(X))


print('Feature functions defined.')

Feature functions defined.


In [5]:
print('Featurizing Tg train  ...', flush=True)
X_tg_raw      = featurize(train_tg['smiles'].tolist())
print('Featurizing Egc train ...', flush=True)
X_egc_raw     = featurize(train_egc['smiles'].tolist())
print('Featurizing Tg test   ...', flush=True)
X_tg_test_raw  = featurize(test_tg['smiles'].tolist())
print('Featurizing Egc test  ...', flush=True)
X_egc_test_raw = featurize(test_egc['smiles'].tolist())

print(f'\nRaw feature shape: {X_tg_raw.shape}')

print('Preprocessing ...')
X_tg,  tg_prep  = build_preprocessor(X_tg_raw)
X_egc, egc_prep = build_preprocessor(X_egc_raw)
X_tg_test  = apply_preprocessor(X_tg_test_raw,  tg_prep)
X_egc_test = apply_preprocessor(X_egc_test_raw, egc_prep)

print(f'Tg  features after cleaning: {X_tg.shape[1]:,}')
print(f'Egc features after cleaning: {X_egc.shape[1]:,}')

Featurizing Tg train  ...
Featurizing Egc train ...
Featurizing Tg test   ...
Featurizing Egc test  ...

Raw feature shape: (4143, 2432)
Preprocessing ...
Tg  features after cleaning: 2,347
Egc features after cleaning: 2,302


In [6]:
import numpy as np
import lightgbm as lgb
import xgboost as xgb

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

SEED = 42


# -----------------------------
# LightGBM parameters
# -----------------------------
def lgbm_params(target_type):
    p = dict(
        objective='regression',
        metric='rmse',
        n_estimators=3000,
        learning_rate=0.015,
        num_leaves=127,
        max_depth=-1,
        min_child_samples=15,
        subsample=0.8,
        colsample_bytree=0.45,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    )

    if target_type == 'egc':
        p['num_leaves'] = 63
        p['min_child_samples'] = 20

    return p


# -----------------------------
# XGBoost parameters
# -----------------------------
def xgb_params(target_type):
    p = dict(
        objective='reg:squarederror',
        n_estimators=3000,
        learning_rate=0.015,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.45,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        tree_method='hist',
        early_stopping_rounds=150  # XGBoost 2.x fix
    )

    if target_type == 'egc':
        p['max_depth'] = 5

    return p


# -----------------------------
# Ensemble training function
# -----------------------------
def train_ensemble(
    X_train,
    y_train,
    X_test,
    target_type,
    n_splits=5
):

    # Ensure numpy arrays
    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train)
    X_test = np.asarray(X_test)

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=SEED
    )

    oof_lgbm = np.zeros(len(X_train))
    oof_xgb = np.zeros(len(X_train))

    test_lgbm = np.zeros(len(X_test))
    test_xgb = np.zeros(len(X_test))

    lp = lgbm_params(target_type)
    xp = xgb_params(target_type)

    for fold, (tr_idx, val_idx) in enumerate(
        kf.split(X_train), 1
    ):

        X_tr = X_train[tr_idx]
        X_val = X_train[val_idx]

        y_tr = y_train[tr_idx]
        y_val = y_train[val_idx]

        # -----------------------------
        # LightGBM
        # -----------------------------
        m_lgbm = lgb.LGBMRegressor(**lp)

        m_lgbm.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=150,
                    verbose=False
                ),
                lgb.log_evaluation(period=0)
            ]
        )

        oof_lgbm[val_idx] = m_lgbm.predict(X_val)
        test_lgbm += (
            m_lgbm.predict(X_test) / n_splits
        )

        # -----------------------------
        # XGBoost
        # -----------------------------
        m_xgb = xgb.XGBRegressor(**xp)

        m_xgb.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        oof_xgb[val_idx] = m_xgb.predict(X_val)
        test_xgb += (
            m_xgb.predict(X_test) / n_splits
        )

        r2_l = r2_score(
            y_val,
            oof_lgbm[val_idx]
        )

        r2_x = r2_score(
            y_val,
            oof_xgb[val_idx]
        )

        print(
            f'Fold {fold} | '
            f'LGBM R²={r2_l:.4f} | '
            f'XGB R²={r2_x:.4f}'
        )

    # -----------------------------
    # Ensemble average
    # -----------------------------
    oof_pred = (
        oof_lgbm + oof_xgb
    ) / 2

    test_pred = (
        test_lgbm + test_xgb
    ) / 2

    r2 = r2_score(
        y_train,
        oof_pred
    )

    print()
    print(f'OOF Ensemble R² = {r2:.4f}')

    return oof_pred, test_pred, r2


# ==========================================
# Train Tg model
# ==========================================

print('=' * 55)
print('Tg (glass transition temperature)')
print('=' * 55)

oof_tg, pred_tg, r2_tg = train_ensemble(
    X_tg,
    y_tg,
    X_tg_test,
    target_type='tg'
)

Tg (glass transition temperature)
Fold 1 | LGBM R²=0.8840 | XGB R²=0.8883
Fold 2 | LGBM R²=0.9013 | XGB R²=0.9100
Fold 3 | LGBM R²=0.8881 | XGB R²=0.8961
Fold 4 | LGBM R²=0.8910 | XGB R²=0.8987
Fold 5 | LGBM R²=0.8739 | XGB R²=0.8810

OOF Ensemble R² = 0.8930


In [7]:
print()

print('=' * 55)
print('Egc (chain band gap)')
print('=' * 55)

oof_egc, pred_egc, r2_egc = train_ensemble(
    X_egc,
    y_egc,
    X_egc_test,
    'egc'
)

print()
print('=' * 55)
print(f'OOF R² Tg   : {r2_tg:.4f}')
print(f'OOF R² Egc  : {r2_egc:.4f}')
print(f'Mean OOF R² : {(r2_tg + r2_egc)/2:.4f}   ← competition metric proxy')
print('=' * 55)


Egc (chain band gap)
Fold 1 | LGBM R²=0.8750 | XGB R²=0.8891
Fold 2 | LGBM R²=0.8976 | XGB R²=0.9046
Fold 3 | LGBM R²=0.9086 | XGB R²=0.9148
Fold 4 | LGBM R²=0.9029 | XGB R²=0.9073
Fold 5 | LGBM R²=0.9174 | XGB R²=0.9226

OOF Ensemble R² = 0.9064

OOF R² Tg   : 0.8930
OOF R² Egc  : 0.9064
Mean OOF R² : 0.8997   ← competition metric proxy


In [8]:
sub_tg          = test_tg[['id']].copy()
sub_tg['target'] = pred_tg

sub_egc          = test_egc[['id']].copy()
sub_egc['target'] = pred_egc

submission = (
    pd.concat([sub_tg, sub_egc], axis=0)
    .sort_values('id')
    .reset_index(drop=True)
)

assert submission.shape[0] == len(test), 'Row count mismatch!'
assert submission['target'].isna().sum() == 0, 'NaN in predictions!'

print('Submission shape:', submission.shape)
print(submission.head(10))

submission.to_csv('/kaggle/working/submission.csv', index=False)

import os

print("Current directory:", os.getcwd())
print("\nFiles in /kaggle/working:")
print(os.listdir('/kaggle/working'))

Submission shape: (4115, 2)
   id      target
0   1  297.882837
1   2    4.957654
2   3   65.694165
3   4   45.435934
4   5   89.138788
5   6  146.887406
6   7   91.483520
7   8  216.284651
8   9  255.663969
9  10    6.355298
Current directory: /kaggle/working

Files in /kaggle/working:
['.virtual_documents', 'oof_scatter.png', 'submission.csv', 'state.db']


In [9]:
check = pd.read_csv('/kaggle/working/submission.csv')
print(check.head())
print(check.shape)

   id      target
0   1  297.882837
1   2    4.957654
2   3   65.694165
3   4   45.435934
4   5   89.138788
(4115, 2)
